# Data Exploration

## Configurações Iniciais

### Configurações a depender do ambiente

In [1]:
import os
import sys
import subprocess

# --- CONFIGURAÇÃO ALVO ---
TARGET_PYSPARK = "4.1.1"

# 1. Identifica o Ambiente (Fora da função para ser global)
IN_COLAB = 'google.colab' in sys.modules
ENV_NAME = "☁️ Google Colab" if IN_COLAB else "💻 Ambiente Local (WSL/Jupyter)"

print(f"Detectado: {ENV_NAME}")
print(f"Versão do Python: {sys.version.split()[0]}")

# 2. Define os Caminhos Globais
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/1_bronze_data/"
    SAVE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/2_silver_data/"
else:
    BASE_PATH = "../1_bronze_data/"
    SAVE_PATH = "../2_silver_data/"
def setup_pyspark():
    if IN_COLAB:
        try:
            import pyspark
            if pyspark.__version__ != TARGET_PYSPARK:
                print(f"(!) Atualizando PySpark de {pyspark.__version__} para {TARGET_PYSPARK}...")
                subprocess.check_call([sys.executable, "-m", "pip", "install", f"pyspark=={TARGET_PYSPARK}", "-q"])
                print("🚨 Reinicie o Ambiente (Runtime > Restart Session) para aplicar a mudança!")
        except ImportError:
            print(f"(!) Instalando PySpark {TARGET_PYSPARK}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", f"pyspark=={TARGET_PYSPARK}", "-q"])

    # Verificação Final
    try:
        import pyspark
        if pyspark.__version__ == TARGET_PYSPARK:
            print(f"✅ PySpark {pyspark.__version__} pronto!")
        else:
            print(f"⚠️ Alerta: PySpark está na versão {pyspark.__version__}. Alvo era {TARGET_PYSPARK}.")
    except ImportError:
        print("❌ Erro: PySpark não encontrado.")

setup_pyspark()

Detectado: 💻 Ambiente Local (WSL/Jupyter)
Versão do Python: 3.12.3
✅ PySpark 4.1.1 pronto!


### Importação de Bibliotecas

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import * # Para definir Schemas (StructType, DoubleType, etc)
from pyspark.sql.window import Window
import pandas as pd
import matplotlib.pyplot as plt

### Inicialização da Sessão Spark

In [3]:
spark = SparkSession.builder \
    .appName("DataExploration") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/05 19:47:38 WARN Utils: Your hostname, PCJULIA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/05 19:47:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/05 19:47:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/05 19:47:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/05 19:47:39 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/05 19:47:39 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


In [4]:
spark

### Importação de Dados

In [5]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .option("encoding", "ISO-8859-1") \
    .load(f"{BASE_PATH}/b_incidentes.csv")

In [6]:
df.show(20)

+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+---------+-------------------+-------+--------------------+-------+-------------+-------------+---------------+----------------+------------+-------------------+-------------------------+---------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|Resolvido|          Encerrado|Duração|  Descrição Resumida|Solução|   Aberto por|Incidente Pai|         Status|Entrou para KPI?|KPI Violado?|Tempo_Pos_Resolução|Resolvido_after_Encerrado|Modified_Record|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+---------+-------------------+-------+--------------------+-------+-------------+-------------+---------------+----------------+------------+-------------------+-------------------------+---------------+
|INC8654273| 3 - Média|   NULL|     NULL|        NULL| 

In [7]:
df.printSchema()

root
 |-- Número: string (nullable = true)
 |-- Prioridade: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Categoria: string (nullable = true)
 |-- Subcategoria: string (nullable = true)
 |-- Grupo designado: string (nullable = true)
 |-- Item de configuração: string (nullable = true)
 |-- Aberto: timestamp (nullable = true)
 |-- Resolvido: timestamp (nullable = true)
 |-- Encerrado: timestamp (nullable = true)
 |-- Duração: integer (nullable = true)
 |-- Descrição Resumida: string (nullable = true)
 |-- Solução: string (nullable = true)
 |-- Aberto por: string (nullable = true)
 |-- Incidente Pai: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Entrou para KPI?: string (nullable = true)
 |-- KPI Violado?: string (nullable = true)
 |-- Tempo_Pos_Resolução: integer (nullable = true)
 |-- Resolvido_after_Encerrado: boolean (nullable = true)
 |-- Modified_Record: boolean (nullable = true)



## Exploração de Dados

### Período Temporal

In [8]:
data_minima_Aberto = df.agg(F.min("Aberto")).first()[0]
data_maxima_Aberto = df.agg(F.max("Aberto")).first()[0]
data_minima_Resolvido = df.agg(F.min("Resolvido")).first()[0]
data_maxima_Resolvido = df.agg(F.max("Resolvido")).first()[0]
data_minima_Encerrado = df.agg(F.min("Encerrado")).first()[0]
data_maxima_Encerrado = df.agg(F.max("Encerrado")).first()[0]
data_minima = min(data_minima_Aberto, data_minima_Resolvido, data_minima_Encerrado)
data_maxima = max(data_maxima_Aberto, data_maxima_Resolvido, data_maxima_Encerrado)
intervalo_total = (data_maxima - data_minima).days
print(f"Data Mínima: {data_minima}")
print(f"Data Máxima: {data_maxima}")
print(f"Intervalo Total: {intervalo_total} dias")

Data Mínima: 2023-01-02 20:19:58
Data Máxima: 2025-12-31 23:45:32
Intervalo Total: 1094 dias


### Incidentes por dia

In [9]:
df = df.withColumn("Data", F.to_date("Aberto"))
df_diario = df.groupby("Data").agg(
    # Coluna Número
    F.count("*").alias("Total_Incidentes"),
    # Coluna Prioridade
    F.count(F.when(F.col("Prioridade") == "1 - Crítica", True)).alias("P1"),
    F.count(F.when(F.col("Prioridade") == "2 - Alta", True)).alias("P2"),
    F.count(F.when(F.col("Prioridade") == "3 - Média", True)).alias("P3"),
    F.count(F.when(F.col("Prioridade") == "4 - Baixa", True)).alias("P4"),
    F.count(F.when(F.col("Prioridade") == "5 - Muito Baixa", True)).alias("P5"),
    # Coluna Produto
    F.count(F.when(F.col("Produto").isNotNull(), True)).alias("Possui_Produto"),
    # Coluna Categoria e Subcategoria - a explorar métricas possíveis
    # Coluna Grupo Designado - a explorar métricas possíveis
    # Coluna Item de configuração - a explorar métricas possíveis
    F.count(F.when(F.col("Item de configuração").isNotNull(), True)).alias("Possui_Item_Configuracao"),
    # Coluna Resolvido
    F.count(F.when(F.col("Resolvido").isNotNull(), True)).alias("Possui_Resolvido"),
    # Coluna Encerrado
    F.count(F.when(F.datediff(F.to_date("Encerrado"), F.to_date("Aberto")) == 0, True)).alias("Encerrados no Mesmo Dia"),
    # Coluna Duração
    F.avg(F.col("Duração")).alias("Duração_Média"),
    # Coluna Código de fechamento - a explorar métricas possíveis
    # Coluna Descrição resumida - a explorar métricas possíveis
    # Coluna Solução
    F.count(F.when(F.col("Solução").isNotNull(), True)).alias("Possui_Solução"),
    F.count(F.when(F.col("Solução") == "Contorno", True)).alias("Solução_Contorno"),
    F.count(F.when(F.col("Solução") == "Definitiva", True)).alias("Solução_Definitiva"),
    # Coluna Aberto por
    F.count(F.when(F.col("Aberto por") == "Manual", True)).alias("Aberto_Manual"),
    F.count(F.when(F.col("Aberto por") == "Monitoramento", True)).alias("Aberto_Monitoramento"),
    # Coluna Incidente Pai
    F.count(F.when(F.col("Incidente Pai").isNotNull(), True)).alias("Possui_Incidente_Pai"),
    # Coluna Status
    F.count(F.when(F.col("Status") == "Encerrado", True)).alias("Encerrado"),
    F.count(F.when(F.col("Status") == "Encerrado Automaticamente", True)).alias("Encerrado_Automaticamente"),
    F.count(F.when(F.col("Status") == "Sem Intervenção", True)).alias("Sem Intervenção"),
    # Entrou para KPI?
    F.count(F.when(F.col("Entrou para KPI?") == "SIM", True)).alias("Entrou_KPI"),
    # KPI Violado?
    F.count(F.when(F.col("KPI Violado?") == "SIM", True)).alias("KPI_Violado"),
    # Coluna Tempo_Pos_Resolução
    F.avg(F.col("Tempo_Pos_Resolução").alias("Tempo_Pos_Resolução_Médio"))
).orderBy("Data")

In [10]:
df_diario.count() == intervalo_total

False

In [11]:
df_diario.count() - intervalo_total

-450

In [12]:
df_datas = spark.createDataFrame(
    [(data_minima, data_maxima)],
    ["start", "end"]
).select(
    F.explode(
        F.sequence(
            F.to_date("start"),
            F.to_date("end")
        )
    ).alias("Data")
)
df_diario.select("Data").printSchema()
df_datas.select("Data").printSchema()
df_diario = df_datas.join(df_diario, on="Data", how="left")
df_diario = df_diario.fillna(0)
df_diario = df_diario.withColumn(
    "tem_incidente",
    F.when(F.col("Total_Incidentes") > 0, 1).otherwise(0)
)

root
 |-- Data: date (nullable = true)

root
 |-- Data: date (nullable = false)



In [13]:
df_diario.count() - intervalo_total - 1

0

In [15]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "browser"


pdf = df_diario.select("Data", "Total_Incidentes")\
    .orderBy("Data")\
    .toPandas()

fig = px.line(
    pdf,
    x="Data",
    y="Total_Incidentes",
    title="Total de Incidentes por Dia",
    markers=True
)

fig.update_layout(
    xaxis_title="Data",
    yaxis_title="Quantidade de Incidentes",
    hovermode="x unified"
)

fig.show()

gio: http://127.0.0.1:45443: Operation not supported
